In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import DataFrame

In [0]:
df_silver = spark.read.table("data_engineering_2026.silver.quality_result")

df_silver.display()

In [0]:

df_summary = df_silver.groupBy("table_name", "run_date").agg(
    max("total_rows").alias("total_rows"),
    sum("issue_count").alias("total_issues"))

df_summary = df_summary.withColumn(
    "quality_score_pct",
    round(((col("total_rows") - col("total_issues")) / col("total_rows")) * 100,2))

df_summary.display()

## 3 Build Column-Level Detail Table

In [0]:
df_column_detail = df_silver.select(
    "run_date",
    "table_name",
    "column_name",
    "check_type",
    "issue_count",
    "total_rows",
    "issue_pct")

df_column_detail\
    .filter(col("issue_count") > 0) \
    .orderBy("table_name", col("issue_count").desc()) \
    .show(50, truncate=False)

## Save Both Gold Tables

In [0]:
df_summary.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("data_engineering_2026.gold.quality_summary") 
  
df_column_detail.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("data_engineering_2026.gold.column_quality_detail") 

In [0]:
df_summary.display()

In [0]:
df_column_detail.display()